In [ ]:
# user settings
input_data_path = ''
output_visualization_figure_folder_path = ''

agent_type_behavior_filter_t1 = 0.9
agent_type_interaction_filter_t2 = 0

In [10]:
# read the data file
import pandas as pd
from collections import defaultdict

log_data = pd.read_csv(input_data_path)

In [11]:
# use AgentMiner to cluster
import pm4py
from pm4py.algo.filtering.dfg import dfg_filtering
import networkx as nx

# for the fragment_id, we delete fragment_idx
def add_fragments_to_events(log_pm4py):
    events_with_fragments_list = []
    for trace in log_pm4py:
        case_id = trace.attributes['concept:name']
        prev_agent_id = None
        fragment_idx = 0
        for evt in trace:
            agent_id = evt['agent_id']
            if (not agent_id==prev_agent_id):
                fragment_idx += 1
                prev_agent_id = agent_id
            evt['fragment_id'] = f"{case_id}_{agent_id}"
    return log_pm4py

def get_agent_model_map(mas_log_df):
    event_concept_name_attr = 'activity_type'
    agent_model_map = {}
    log_without_fragment = pm4py.convert_to_event_log(mas_log_df)
    log_with_fragment = add_fragments_to_events(log_without_fragment)
    new_mas_log_df = pm4py.convert_to_dataframe(log_with_fragment)
    log_groupby_agent = new_mas_log_df.groupby(['agent_id'],sort=False)
    for agent_id,agent_log_df in log_groupby_agent:
        df0 = pm4py.format_dataframe(agent_log_df, case_id='fragment_id', activity_key=event_concept_name_attr, timestamp_key='timestamp')
        agent_log_pm4py = pm4py.convert_to_event_log(df0)
        dfg_pm4py, start_activities, end_activities = pm4py.discover_directly_follows_graph(agent_log_pm4py)
        activities_count = pm4py.get_event_attribute_values(agent_log_pm4py, "concept:name")
        activity_frequency_filter = 1
        dfg_pm4py, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_activities_percentage(
            dfg_pm4py, 
            start_activities, 
            end_activities, 
            activities_count, 
            activity_frequency_filter
            )
        total_event_count = 0

        for activity_type in activities_count:
            total_event_count += activities_count[activity_type]
        dfg_obj = {
            'agent_types':{agent_id},
            'event_count': total_event_count,
            'activity_count': len(activities_count),
            'flow_count': len(dfg_pm4py),
            'activity_types': activities_count,
            'control_flows': dfg_pm4py,
            'input_activity_types': start_activities,
            'output_activity_types': end_activities
        }
        agent_model_map[agent_id] = dfg_obj
    return agent_model_map

def get_dfg_normalized_graph_edit_distance_artem(dfg1_obj, dfg2_obj):

    e_intersect = [e for e in dfg1_obj['control_flows'] if e in dfg2_obj['control_flows']]
    e_intersect.extend([('_i_',i) for i in dfg1_obj['input_activity_types'] if i in dfg2_obj['input_activity_types']])
    e_intersect.extend([(o,'_o_') for o in dfg1_obj['output_activity_types'] if o in dfg2_obj['output_activity_types']])
    e_count_intersect = len(e_intersect)
    e_count_dfg1 = len(dfg1_obj['control_flows']) + len(dfg1_obj['input_activity_types']) + len(dfg1_obj['output_activity_types'])
    e_count_dfg2 = len(dfg2_obj['control_flows']) + len(dfg2_obj['input_activity_types']) + len(dfg2_obj['output_activity_types'])
    if e_count_dfg1>0 and e_count_dfg2>0:
        dist = 1 - max(e_count_intersect/e_count_dfg1,e_count_intersect/e_count_dfg2)
    else:
        dist = 0
    return (dist,e_count_intersect)

def get_distance_matrix(agent_dfg_map):
    # calculate DFG distance among agent instances in form of a map (agent1,agent2) -> distance_betwen_agent1_agent2
    dfg_dist_map = {}
    dfg_intersect_map = {}
    agent_with_dedicated_activities_list = []
    for agent1_id in agent_dfg_map:
        dfg1_total_intersect=0
        for agent2_id in agent_dfg_map:
            ged = None
            intersect_count = None
            if (agent2_id,agent1_id) in dfg_dist_map:
                ged = dfg_dist_map[(agent2_id,agent1_id)]
                intersect_count = dfg_intersect_map[(agent2_id,agent1_id)]
            elif agent1_id==agent2_id:
                ged = 0
                intersect_count = -1
            else:
                dfg1_obj = agent_dfg_map[agent1_id]
                dfg2_obj = agent_dfg_map[agent2_id]
                ged,intersect_count = get_dfg_normalized_graph_edit_distance_artem(dfg1_obj,dfg2_obj)
            dfg_dist_map[(agent1_id,agent2_id)] = ged
            dfg_intersect_map[(agent1_id,agent2_id)] = intersect_count
            dfg1_total_intersect=dfg1_total_intersect+intersect_count
        if dfg1_total_intersect<0:
            print("___ Agent with dedicated activities: ", agent1_id)    
            agent_with_dedicated_activities_list.append(agent1_id)
    return (dfg_dist_map,dfg_intersect_map,agent_with_dedicated_activities_list)

def group_clusters(dfg_dist_map,max_dist=0.99):
    # build agent dfg distance graph (nodes are agent instances, edge weights are distances between agent DFGs)
    addg_nx = nx.DiGraph()
    for a1,a2 in dfg_dist_map:
        if not addg_nx.has_node(a1): addg_nx.add_node(a1)
        if not addg_nx.has_node(a2): addg_nx.add_node(a2)
        if (not a1==a2) and (not dfg_dist_map[(a1,a2)] > max_dist):
            if not addg_nx.has_edge(a1,a2):
                addg_nx.add_edge(a1,a2,weight=dfg_dist_map[(a1,a2)])

    if sum(data['weight'] for _, _, data in addg_nx.edges(data=True)) > 0 and len(addg_nx.edges) > 4:
        agent_clusters_list = nx.algorithms.community.greedy_modularity_communities(addg_nx,weight='weight')
    else:
        agent_clusters_list = [{a} for a in addg_nx.nodes]
    inst_cluster_map = {}
    cl_idx = 1
    for cl_inst_set in agent_clusters_list:
        if len(cl_inst_set) >= 1:
            cluster_id = 'A'+str(cl_idx)
            cl_idx += 1

        for inst_id in cl_inst_set:
            inst_cluster_map[inst_id] = cluster_id

    return inst_cluster_map, addg_nx

def run_AgentMiner(log_df, threshold):
    log_df_1 = pm4py.format_dataframe(log_df, case_id='case_id', activity_key='activity_type', timestamp_key='timestamp')
    agent_model_map = get_agent_model_map(log_df_1)
    dfg_dist_map,dfg_intersect_map,agent_with_dedicated_activities_list = get_distance_matrix(agent_model_map)
    inst_cluster_map, addg_nx = group_clusters(dfg_dist_map, threshold)
    final_inst_cluster_map = {}
    for resource_tuple in inst_cluster_map:
        if type(resource_tuple) == tuple:
            resource = resource_tuple[0]
            final_inst_cluster_map[resource] = inst_cluster_map[resource_tuple]
    if final_inst_cluster_map == {}:
        final_inst_cluster_map = inst_cluster_map
    return final_inst_cluster_map

def add_clusters_to_log(inst_cluster_map, log_df):
    def make_agent_evt(evt):
        inst_id = evt['agent_id']
        cluster_id = inst_cluster_map[inst_id]
        evt['agent_inst_id'] = inst_id
        evt['agent_id'] = cluster_id
        evt['agent_activity_type'] = f"{cluster_id}|{evt['activity_type']}"
        return evt
    log_df_test = log_df.apply(make_agent_evt,axis=1)
    return log_df_test

In [12]:
# get the C/I, HI/HC information

def compute_CI_frequencies_with_agents(df):
    """
    Computes:
    1) Global Continuation (C) and Interruption (I) frequencies
    2) Per-agent Continuation (C) and Interruption (I) frequencies

    Returns:
        global_freq: dict
        agent_freq: dict
    """
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Sort by agent and time
    df = df.sort_values(['agent_id', 'timestamp'])

    # ---------- GLOBAL ----------
    global_freq = defaultdict(lambda: {'C': 0, 'I': 0})

    # ---------- PER AGENT ----------
    # structure:
    # { agent_id : { (act_i, act_j): {'C':x, 'I':y} } }
    agent_freq = {}

    for agent, agent_df in df.groupby('agent_id'):

        agent_df = agent_df.sort_values('timestamp').reset_index(drop=True)

        # initialize dictionary for this agent
        agent_freq[agent] = defaultdict(lambda: {'C': 0, 'I': 0})

        for i in range(len(agent_df) - 1):

            e_i = agent_df.iloc[i]
            e_j = agent_df.iloc[i + 1]

            pair = (e_i['activity_type'], e_j['activity_type'])

            if e_i['case_id'] == e_j['case_id']:
                # Continuation
                global_freq[pair]['C'] += 1
                agent_freq[agent][pair]['C'] += 1
            else:
                # Interruption
                global_freq[pair]['I'] += 1
                agent_freq[agent][pair]['I'] += 1

    # Convert defaultdicts to normal dicts
    global_freq = {k: dict(v) for k, v in global_freq.items()}
    agent_freq = {
        agent: {k: dict(v) for k, v in pairs.items()}
        for agent, pairs in agent_freq.items()
    }

    return global_freq, agent_freq

In [13]:
# construct DFG
from pm4py.objects.conversion.dfg import converter as dfg_mining
import pm4py.visualization.dfg.visualizer as dfg_vis
from collections import defaultdict

def discover_dfg_pm4py(dfg_id,log_pm4py,activity_frequency_filter):
    dfg_pm4py, start_activities, end_activities = pm4py.discover_directly_follows_graph(log_pm4py)
    activities_count = pm4py.get_event_attribute_values(log_pm4py, "concept:name")
    dfg_pm4py, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_activities_percentage(
        dfg_pm4py, 
        start_activities, 
        end_activities, 
        activities_count, 
        activity_frequency_filter
        )
    total_event_count = 0
    for activity_type in activities_count:
        total_event_count += activities_count[activity_type]
    dfg_obj = {
        'agent_types':{dfg_id},
        'event_count': total_event_count,
        'activity_count': len(activities_count),
        'flow_count': len(dfg_pm4py),
        'activity_types': activities_count,
        'control_flows': dfg_pm4py,
        'input_activity_types': start_activities,
        'output_activity_types': end_activities
    }

    return dfg_obj

def discover_pn_from_dfg(agent_dfg_obj):
    pn_pm4py, im_pm4py, fm_pm4py = dfg_mining.apply(
        agent_dfg_obj['control_flows'],
        parameters = {'start_activities':agent_dfg_obj['input_activity_types'],'end_activities':agent_dfg_obj['output_activity_types']},
        variant=pm4py.objects.conversion.dfg.converter.Variants.VERSION_TO_PETRI_NET_INVISIBLES_NO_DUPLICATES)    
    return (pn_pm4py, im_pm4py, fm_pm4py)


def discover_agent_model(agent_id,agent_log_pm4py,freq_filter,agent_xes_log_name=None):
    a_dfg_obj = discover_dfg_pm4py(agent_id,agent_log_pm4py,freq_filter)

    # added for visualize dfg
    viz = dfg_vis.apply(a_dfg_obj['control_flows'], agent_log_pm4py, a_dfg_obj['activity_types'], parameters={
            dfg_vis.Variants.FREQUENCY.value.Parameters.START_ACTIVITIES: a_dfg_obj['input_activity_types'],
            dfg_vis.Variants.FREQUENCY.value.Parameters.END_ACTIVITIES: a_dfg_obj['output_activity_types'],})
    
    dfg_vis.save(viz, f"{output_visualization_figure_folder_path}/{agent_id}_agent_type_behavioral_DFG.png")

    return a_dfg_obj


def compute_agent_resource_merged(log):

    merged_counts = defaultdict(lambda: defaultdict(lambda: {
        "agent_counts": [0,0,0,0],     # cont_same_res, cont_diff_res, intr_same_res, intr_diff_res
        "resource_counts": [0,0,0,0]   # cont_C, cont_I, intr_C, intr_I
    }))

    # modify here
    log = log.sort_values(["case_id","timestamp"]).reset_index(drop=True)
    # log = log.sort_values(["case_id"]).reset_index(drop=True)

    agents = log["agent_id"].tolist()
    acts = log["activity_type"].tolist()
    res = log["agent_inst_id"].tolist()
    cases = log["case_id"].tolist()
    ts = log["timestamp"].tolist()

    n = len(log)

    # index resource events
    resource_events = defaultdict(list)
    for i,r in enumerate(res):
        resource_events[r].append(i)

    def resource_interrupted(r, t1, t2, case):
        for k in resource_events[r]:
            if ts[k] > t1 and ts[k] < t2 and cases[k] != case:
                return True
        return False

    # ---- per case ----
    for case_id, case_df in log.groupby("case_id"):

        idx = case_df.index.tolist()
        a = case_df["agent_id"].tolist()
        ac = case_df["activity_type"].tolist()
        r = case_df["agent_inst_id"].tolist()
        t = case_df["timestamp"].tolist()

        m = len(case_df)

        # ==============================
        # AGENT CONTINUATION
        # ==============================
        for i in range(m-1):

            if a[i] == a[i+1]:

                pair = (ac[i], ac[i+1])
                agent = a[i]

                same_res = r[i] == r[i+1]

                gi, gj = idx[i], idx[i+1]

                if same_res:
                    merged_counts[agent][pair]["agent_counts"][0] += 1

                    if resource_interrupted(r[i], t[i], t[i+1], case_id):
                        merged_counts[agent][pair]["resource_counts"][1] += 1
                    else:
                        merged_counts[agent][pair]["resource_counts"][0] += 1

                else:
                    merged_counts[agent][pair]["agent_counts"][1] += 1


        # ==============================
        # AGENT INTERRUPTION
        # ==============================
        i = 0
        while i < m-1:

            agent = a[i]

            j = i
            while j+1 < m and a[j+1] == agent:
                j += 1

            last_act = ac[j]
            last_res = r[j]

            k = j+1
            while k < m and a[k] != agent:
                k += 1

            if k < m and k > j+1:

                pair = (last_act, ac[k])
                same_res = last_res == r[k]

                if same_res:

                    merged_counts[agent][pair]["agent_counts"][2] += 1

                    if resource_interrupted(last_res, t[j], t[k], case_id):
                        merged_counts[agent][pair]["resource_counts"][3] += 1
                    else:
                        merged_counts[agent][pair]["resource_counts"][2] += 1

                else:
                    merged_counts[agent][pair]["agent_counts"][3] += 1

            i = j+1


    # convert to normal dict
    final = {}

    for agent,pairs in merged_counts.items():

        final[agent] = {}

        for pair,vals in pairs.items():

            final[agent][pair] = {
                "agent_counts": tuple(vals["agent_counts"]),
                "resource_counts": tuple(vals["resource_counts"])
            }

    return final 


def merge_agent_pair_counts(agent_petri_model_map, merged_work_counts):
    """
    Merge agent DFG control flows with agent/resource level counts.

    Inputs:
        agent_petri_model_map: dict
            {agent: {'control_flows': {('act_i','act_j'): freq, ...}, ...}}
        merged_work_counts: dict
            {agent: {(act_i, act_j): {'agent_counts': (...), 'resource_counts': (...)} } }

    Returns:
        result: dict
            {agent: {('act_i','act_j'): {'control_flow': freq, 
                                         'agent_counts': (...),
                                         'resource_counts': (...)} } }
    """
    result = {}

    for agent, model in agent_petri_model_map.items():
        result[agent] = {}
        control_flows = model.get("control_flows", {})

        agent_counts_dict = merged_work_counts.get(agent, {})

        for pair, freq in control_flows.items():
            counts = agent_counts_dict.get(pair, {"agent_counts": (0,0,0,0),
                                                 "resource_counts": (0,0,0,0)})
            result[agent][pair] = {
                "control_flow": freq,
                "agent_counts": counts["agent_counts"],
                "resource_counts": counts["resource_counts"]
            }

    return result

def get_agent_model(cluster_mas_log_df, activity_type, freq_filter):
    agent_petri_model_map = {}
    agent_CI_dic = compute_agent_resource_merged(cluster_mas_log_df)
    log_groupby_agent = cluster_mas_log_df.groupby(['agent_id'],sort=False)
    for agent_id,agent_log_df in log_groupby_agent:
        if type(agent_id) == tuple:
            agent_id = agent_id[0]
        df0 = pm4py.format_dataframe(agent_log_df, case_id='fragment_id', activity_key=activity_type, timestamp_key='timestamp')
        agent_log_pm4py = pm4py.convert_to_event_log(df0)

        cl_dfg_obj = discover_agent_model(agent_id,agent_log_pm4py,freq_filter,agent_xes_log_name=None)
        agent_petri_model_map[agent_id] = cl_dfg_obj

    merge_results = merge_agent_pair_counts(agent_petri_model_map, agent_CI_dic)
    return agent_petri_model_map, agent_CI_dic, merge_results

In [14]:
import copy
from collections import defaultdict
from graphviz import Digraph

# get the agent interaction relationship
def create_interaction_log(mas_log_df,agent_trace_col_name='fragment_id'):
    agent_traces_over_mas_df = pm4py.format_dataframe(mas_log_df, case_id=agent_trace_col_name, activity_key='agent_id', timestamp_key='timestamp')
    agent_traces_over_mas_pm4py = pm4py.convert_to_event_log(agent_traces_over_mas_df)
    hn_events_list = [f_trace[0] for f_trace in agent_traces_over_mas_pm4py]
    in_df = pd.DataFrame(hn_events_list)
    in_df = pm4py.format_dataframe(in_df, case_id='case_id', activity_key='agent_id', timestamp_key='timestamp')
    
    return pm4py.convert_to_event_log(in_df)

def discover_interaction_dfg_pm4py(log_pm4py,activity_frequency_filter):
    # make a copy so original log is unchanged
    log_agents = copy.deepcopy(log_pm4py)

    # replace activity with agent_id
    for trace in log_agents:
        for event in trace:
            event["concept:name"] = event["agent_activity_type"]
            
    dfg_pm4py, start_activities, end_activities = pm4py.discover_directly_follows_graph(log_agents)
    activities_count = pm4py.get_event_attribute_values(log_agents, "concept:name")
    print(activities_count)
    dfg_pm4py, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_activities_percentage(
        dfg_pm4py, 
        start_activities, 
        end_activities, 
        activities_count, 
        activity_frequency_filter
        )
    total_event_count = 0
    for activity_type in activities_count:
        total_event_count += activities_count[activity_type]
    dfg_obj = {
        'event_count': total_event_count,
        'activity_count': len(activities_count),
        'flow_count': len(dfg_pm4py),
        'activity_types': activities_count,
        'control_flows': dfg_pm4py,
        'input_activity_types': start_activities,
        'output_activity_types': end_activities
    }

    return dfg_obj


def compute_handover_idle_busy(log_df):

    agents = log_df["agent_id"].tolist()
    acts = log_df["agent_activity_type"].tolist()
    cases = log_df["case_id"].tolist()
    resources = log_df["agent_inst_id"].tolist()
    ts = log_df["timestamp"].tolist()

    n = len(log_df)

    # index events per resource for fast lookup
    resource_events = defaultdict(list)
    for i, r in enumerate(resources):
        resource_events[r].append(i)

    result = defaultdict(lambda: {"HI":0, "HB":0})

    # process per case
    for case_id, case_df in log_df.groupby("case_id"):

        idx = case_df.index.tolist()
        m = len(idx)

        for i in range(m-1):

            gi = idx[i]
            gj = idx[i+1]

            Ai = agents[gi]
            Aj = agents[gj]

            # only if agent changes
            if Ai == Aj:
                continue

            Ri = resources[gi]
            Rj = resources[gj]

            ti = ts[gi]
            tj = ts[gj]

            pair = (acts[gi], acts[gj])

            busy = False

            # check events of Rj between ti and tj
            for k in resource_events[Rj]:
                if ts[k] >= ti and ts[k] <= tj:
                    if cases[k] != case_id:
                        busy = True
                        break

            if busy:
                result[pair]["HB"] += 1
            else:
                result[pair]["HI"] += 1

    # convert to simple dict
    final = {}
    for pair,v in result.items():
        final[pair] = (v["HI"], v["HB"])

    return final


def first_filter_interaction_pairs_by_agent_activity(agent_petri_model_map, pair_dict):

    # Step 1: build valid agent|activity keys
    valid_keys = set()

    for agent, model in agent_petri_model_map.items():
        for activity in model.get("activity_types", {}):
            valid_keys.add(f"{agent}|{activity}")

    # Step 2: filter pairs
    second_filtered = {}

    for (src, tgt), value in pair_dict.items():
        if src in valid_keys and tgt in valid_keys:
            second_filtered[(src, tgt)] = value

    return second_filtered


def filter_handover_by_threshold(handover_dict, threshold):
    """
    threshold: value between 0 and 1
    """

    # compute total frequencies
    totals = {k: v[0] + v[1] for k, v in handover_dict.items()}

    total_sum = sum(totals.values())

    # filter
    filtered = {}

    for k, v in handover_dict.items():
        freq = v[0] + v[1]
        ratio = freq / total_sum

        if ratio >= threshold:
            filtered[k] = v

    return filtered

def collapse_interaction_to_agent_level(interaction_obj):

    # --- activity_types → agent_types ---
    agent_types = defaultdict(int)

    for key, count in interaction_obj["activity_types"].items():
        agent = key.split("|")[0]
        agent_types[agent] += count

    # --- control flows ---
    agent_flows = defaultdict(int)

    for (src, tgt), count in interaction_obj["control_flows"].items():
        src_agent = src.split("|")[0]
        tgt_agent = tgt.split("|")[0]

        agent_flows[(src_agent, tgt_agent)] += count

    # --- start activities ---
    start_agents = defaultdict(int)

    for key, count in interaction_obj["input_activity_types"].items():
        agent = key.split("|")[0]
        start_agents[agent] += count

    # --- end activities ---
    end_agents = defaultdict(int)

    for key, count in interaction_obj["output_activity_types"].items():
        agent = key.split("|")[0]
        end_agents[agent] += count

    # --- build new object ---
    new_obj = {
        "event_count": interaction_obj["event_count"],
        "activity_count": len(agent_types),
        "flow_count": len(agent_flows),
        "activity_types": dict(agent_types),
        "control_flows": dict(agent_flows),
        "input_activity_types": dict(start_agents),
        "output_activity_types": dict(end_agents)
    }

    return new_obj
    
def get_interaction_model(cluster_mas_log_df, agent_petri_model_map, threshold):
    handover_IB_dic = compute_handover_idle_busy(cluster_mas_log_df)

    first_filtered_handover_IB_dic = first_filter_interaction_pairs_by_agent_activity(agent_petri_model_map, handover_IB_dic)

    second_filtered_handover_IB_dic = filter_handover_by_threshold(first_filtered_handover_IB_dic, threshold)

    return second_filtered_handover_IB_dic


def build_agent_interaction_model(model):

    agent_model = {
        "activity_types": defaultdict(int),
        "control_flows": defaultdict(int),
        "handover_IB": defaultdict(lambda: [0,0]),
        "input_activity_types": defaultdict(int),
        "output_activity_types": defaultdict(int)
    }

    # activity types
    for act, freq in model["activity_types"].items():
        agent = act.split("|")[0]
        agent_model["activity_types"][agent] += freq

    # control flows
    for (src, tgt), freq in model["control_flows"].items():

        src_agent = src.split("|")[0]
        tgt_agent = tgt.split("|")[0]

        agent_model["control_flows"][(src_agent, tgt_agent)] += freq

    # handover idle/busy
    for (src, tgt), (hi, hb) in model["handover_IB"].items():

        src_agent = src.split("|")[0]
        tgt_agent = tgt.split("|")[0]

        agent_model["handover_IB"][(src_agent, tgt_agent)][0] += hi
        agent_model["handover_IB"][(src_agent, tgt_agent)][1] += hb

    # input activities
    for act, freq in model["input_activity_types"].items():
        agent = act.split("|")[0]
        agent_model["input_activity_types"][agent] += freq

    # output activities
    for act, freq in model["output_activity_types"].items():
        agent = act.split("|")[0]
        agent_model["output_activity_types"][agent] += freq

    # convert handover lists to tuples
    agent_model["handover_IB"] = {
        k: tuple(v) for k,v in agent_model["handover_IB"].items()
    }

    return agent_model


# combine this handover_IB_dic with the agent_petri_model_map
def integrate_agent_with_interaction_model(agent_petri_model_map, handover_IB_dic):
    """
    Combine multiple agent Petri models with interaction (handover) info
    into a single structure with prefixed activities and combined flows.
    
    Parameters
    ----------
    agent_petri_model_map : dict
        Mapping from agent ID to its Petri model summary, e.g., {'A1': {...}, 'A2': {...}}
    handover_IB_dic : dict
        Dictionary containing handover information between agents.
        Format: {('A1|act', 'A2|act'): (handover_i_count, handover_b_count), ...}

    Returns
    -------
    integrated : dict
        Combined structure with prefixed activities, merged counts and flows, including handovers as tuples.
    """
    integrated_mas_dfg = {
        'activity_types': {},        # combined activity counts
        'control_flows': {},         # combined flows
        'handover_IB': {},  # store tuples for handovers
        'input_activity_types': {},  # combined start activities
        'output_activity_types': {}  # combined end activities
    }

    # Add each agent's activities and flows
    for agent, model in agent_petri_model_map.items():
        # Prefix activities with agent
        for act, count in model['activity_types'].items():
            key = f"{agent}|{act}"
            integrated_mas_dfg['activity_types'][key] = integrated_mas_dfg['activity_types'].get(key, 0) + count

        # Prefix control flows
        for (src, tgt), count in model['control_flows'].items():
            src_key = f"{agent}|{src}"
            tgt_key = f"{agent}|{tgt}"
            integrated_mas_dfg['control_flows'][(src_key, tgt_key)] = integrated_mas_dfg['control_flows'].get((src_key, tgt_key), 0) + count

        # Merge input_activity_types
        for act, count in model.get('input_activity_types', {}).items():
            key = f"{agent}|{act}"
            integrated_mas_dfg['input_activity_types'][key] = integrated_mas_dfg['input_activity_types'].get(key, 0) + count

        # Merge output_activity_types
        for act, count in model.get('output_activity_types', {}).items():
            key = f"{agent}|{act}"
            integrated_mas_dfg['output_activity_types'][key] = integrated_mas_dfg['output_activity_types'].get(key, 0) + count

    # Add handover interactions
    for (src_key, tgt_key), (handover_i_count, handover_b_count) in handover_IB_dic.items():
        # Add the handover tuple
        integrated_mas_dfg['handover_IB'][(src_key, tgt_key)] = (handover_i_count, handover_b_count)

        # Add the handover as a control flow with frequency
        integrated_mas_dfg['control_flows'][(src_key, tgt_key)] = integrated_mas_dfg['control_flows'].get((src_key, tgt_key), 0) + handover_i_count + handover_b_count


    # visualization

    viz = dfg_vis.apply(integrated_mas_dfg['control_flows'], None, integrated_mas_dfg['activity_types'], parameters={
        dfg_vis.Variants.FREQUENCY.value.Parameters.START_ACTIVITIES: integrated_mas_dfg['input_activity_types'],
        dfg_vis.Variants.FREQUENCY.value.Parameters.END_ACTIVITIES: integrated_mas_dfg['output_activity_types'],})
    
    # normal MAS_DFG
    # dfg_vis.save(viz, f"/Users/qingtan/Desktop/DFG_discovery/newest_dfg_file/MAS_DFG.pdf")

    # agent interaction model visualziation
    viz_interaction_obj = build_agent_interaction_model(integrated_mas_dfg)
    dfg_no_self = {edge: freq for edge, freq in viz_interaction_obj['control_flows'].items() if edge[0] != edge[1]}
    dfg_self = {edge[0]: freq for edge, freq in viz_interaction_obj['control_flows'].items() if edge[0] == edge[1]}


    viz_interaction = dfg_vis.apply(dfg_no_self, None, viz_interaction_obj['activity_types'], parameters={
            dfg_vis.Variants.FREQUENCY.value.Parameters.START_ACTIVITIES: viz_interaction_obj['input_activity_types'],
            dfg_vis.Variants.FREQUENCY.value.Parameters.END_ACTIVITIES: viz_interaction_obj['output_activity_types'],})
    
    viz_interaction.format = "pdf"
    dfg_vis.save(viz_interaction, f"{output_visualization_figure_folder_path}/agent_type_interaction_DFG.pdf")


    def visualize_mas_dfg(integrated_mas_dfg, output_path):

        g = Digraph("MAS_DFG", format="png")

        # start and end nodes
        g.node("START", shape="circle", style="filled", fillcolor="white")
        g.node("END", shape="doublecircle", style="filled", fillcolor="white")

        # add activity nodes
        for act, freq in integrated_mas_dfg['activity_types'].items():
            g.node(act, f"{act}\n({freq})", shape="box")

        # add control flows
        for (src, tgt), freq in integrated_mas_dfg['control_flows'].items():

            src_agent = src.split('|')[0]
            tgt_agent = tgt.split('|')[0]

            if src_agent == tgt_agent:
                color = "blue"
            else:
                color = "red"

            g.edge(src, tgt, label=str(freq), color=color)

        # add start edges
        for act, freq in integrated_mas_dfg['input_activity_types'].items():
            g.edge("START", act, label=str(freq), color="black")

        # add end edges
        for act, freq in integrated_mas_dfg['output_activity_types'].items():
            g.edge(act, "END", label=str(freq), color="black")

        g.render(output_path, format='pdf', view=False)


    def visualize_agent_interaction_handover(integrated_mas_dfg, output_path):
        g = Digraph("MAS_Handover", format="png")

        # find nodes involved in cross-agent arcs
        used_nodes = set()
        cross_agent_edges = []

        for (src, tgt), freq in integrated_mas_dfg['control_flows'].items():
            src_agent = src.split('|')[0]
            tgt_agent = tgt.split('|')[0]

            if src_agent != tgt_agent:
                used_nodes.add(src)
                used_nodes.add(tgt)
                cross_agent_edges.append((src, tgt, freq))

        # add only nodes that appear in cross-agent arcs
        for act in used_nodes:
            freq = integrated_mas_dfg['activity_types'].get(act, 0)
            g.node(act, f"{act}\n({freq})", shape="box")

        # add only cross-agent edges
        for src, tgt, freq in cross_agent_edges:
            if (src, tgt) in integrated_mas_dfg['handover_IB']:
                hi, hb = integrated_mas_dfg['handover_IB'][(src, tgt)]
                label = f"HI:{hi} | HB:{hb}"
            else:
                label = str(freq)

            g.edge(src, tgt, label=label, color="red")

        g.render(output_path, format="pdf", view=False)

    # MAS_DFG with visualization
    visualize_mas_dfg(integrated_mas_dfg, f'{output_visualization_figure_folder_path}/MAS_DFG')
    visualize_agent_interaction_handover(integrated_mas_dfg,  f'{output_visualization_figure_folder_path}/inter_agent_type_handover_visualization')
    
    return integrated_mas_dfg


def draw_multiple_agent_models(agent_merge_results):
    for agent, flows in agent_merge_results.items():
        # get event continuation with dashed and solid
        g = Digraph(f"{agent}_model", format="png")

        activities = set()

        # collect activities
        for (src, tgt) in flows.keys():
            activities.add(src)
            activities.add(tgt)

        # add nodes
        for act in activities:
            node = f"{agent}|{act}"
            g.node(node, shape="box", label=node)

        # add edges
        for (src, tgt), info in flows.items():

            src_node = f"{agent}|{src}"
            tgt_node = f"{agent}|{tgt}"

            cont_same, cont_diff, intr_same, intr_diff = info["agent_counts"]

            # continuation edge
            if cont_same > 0 or cont_diff > 0:
                label = f"{cont_same+cont_diff}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="solid"
                )

            # interruption edge
            if intr_same > 0 or intr_diff > 0:
                label = f"{intr_same+intr_diff}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="dashed",
                    color='red'
                )

        output_path = f"{output_visualization_figure_folder_path}/{agent}_intra_agent_type_consecutive_and_handback_visualization"
        g.render(output_path, format='pdf', view=False)

        # get labels with S|D for solid
        g = Digraph(f"{agent}_model", format="png")

        activities = set()

        # collect activities
        for (src, tgt) in flows.keys():
            activities.add(src)
            activities.add(tgt)

        # add nodes
        for act in activities:
            node = f"{agent}|{act}"
            g.node(node, shape="box", label=node)

        # add edges
        for (src, tgt), info in flows.items():

            src_node = f"{agent}|{src}"
            tgt_node = f"{agent}|{tgt}"

            cont_same, cont_diff, intr_same, intr_diff = info["agent_counts"]

            # continuation edge
            if cont_same > 0 or cont_diff > 0:
                label = f"S:{cont_same}|D:{cont_diff}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="solid"
                )

        output_path = f"{output_visualization_figure_folder_path}/{agent}_intra_agent_type_task_transfer_visualization_(consecutive)"
        g.render(output_path, format='pdf', view=False)
        
        # get labels with S|D for dashed
        g = Digraph(f"{agent}_model", format="png")

        activities = set()

        # collect activities
        for (src, tgt) in flows.keys():
            activities.add(src)
            activities.add(tgt)

        # add nodes
        for act in activities:
            node = f"{agent}|{act}"
            g.node(node, shape="box", label=node)

        # add edges
        for (src, tgt), info in flows.items():

            src_node = f"{agent}|{src}"
            tgt_node = f"{agent}|{tgt}"

            cont_same, cont_diff, intr_same, intr_diff = info["agent_counts"]

            # interruption edge
            if intr_same > 0 or intr_diff > 0:
                label = f"S:{intr_same}|D:{intr_diff}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="dashed",
                    color='red'
                )

        output_path = f"{output_visualization_figure_folder_path}/{agent}_intra_agent_type_task_transfer_visualization_(handback)"
        g.render(output_path, format='pdf', view=False)

        # get resource_level C/I for S, solid
        g = Digraph(f"{agent}_model", format="png")

        activities = set()

        # collect activities
        for (src, tgt) in flows.keys():
            activities.add(src)
            activities.add(tgt)

        # add nodes
        for act in activities:
            node = f"{agent}|{act}"
            g.node(node, shape="box", label=node)

        # add edges
        for (src, tgt), info in flows.items():

            src_node = f"{agent}|{src}"
            tgt_node = f"{agent}|{tgt}"

            cont_res_C, cont_res_I, intr_res_C, intr_res_I = info["resource_counts"]

            # continuation edge
            if cont_res_C > 0 or cont_res_I > 0:
                label = f"C:{cont_res_C}|I:{cont_res_I}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="solid"
                )

        output_path = f"{output_visualization_figure_folder_path}/{agent}_intra_agent_type_continuation_and_interruption_visualization_(consecutive)"
        g.render(output_path, format='pdf', view=False)

        # get resource_level C/I for S, dash
        g = Digraph(f"{agent}_model", format="png")

        activities = set()

        # collect activities
        for (src, tgt) in flows.keys():
            activities.add(src)
            activities.add(tgt)

        # add nodes
        for act in activities:
            node = f"{agent}|{act}"
            g.node(node, shape="box", label=node)

        # add edges
        for (src, tgt), info in flows.items():

            src_node = f"{agent}|{src}"
            tgt_node = f"{agent}|{tgt}"

            cont_res_C, cont_res_I, intr_res_C, intr_res_I = info["resource_counts"]

            if intr_res_C > 0 or intr_res_I > 0:
                label = f"C:{intr_res_C}|I:{intr_res_I}"
                g.edge(
                    src_node,
                    tgt_node,
                    label=label,
                    style="dashed",
                    color='red'
                )

        output_path = f"{output_visualization_figure_folder_path}/{agent}_intra_agent_type_continuation_and_interruption_visualization_(handback)"
        g.render(output_path, format='pdf', view=False)

In [15]:
inst_cluster_map = run_AgentMiner(log_data, 1.0)
log_df_test = add_clusters_to_log(inst_cluster_map, log_data)
log_df = pm4py.format_dataframe(log_df_test, case_id='case_id', activity_key='activity_type', timestamp_key='timestamp')
log_pm4py = pm4py.convert_to_event_log(log_df)
log_pm4py = add_fragments_to_events(log_pm4py)
log_df = pm4py.convert_to_dataframe(log_pm4py)

agent_petri_model_map, agent_CI_dic, merge_results = get_agent_model(log_df, 'activity_type', freq_filter = (1 - agent_type_behavior_filter_t1))
second_filtered_handover_IB_dic = get_interaction_model(log_df, agent_petri_model_map, agent_type_interaction_filter_t2)
integrated_mas_dfg = integrate_agent_with_interaction_model(agent_petri_model_map, second_filtered_handover_IB_dic)
draw_multiple_agent_models(merge_results)

/Users/qingtan/anaconda3/lib/python3.11/site-packages/pm4py/objects/log/util/dataframe_utils.py:177: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], utc=True)
/Users/qingtan/anaconda3/lib/python3.11/site-packages/pm4py/objects/log/util/dataframe_utils.py:177: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], utc=True)
/Users/qingtan/anaconda3/lib/python3.11/site-packages/pm4py/objects/log/util/dataframe_utils.py:177: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], utc=True)
/Users/